In [1]:
import pandas as pd 
import pickle 
from sklearn.metrics import confusion_matrix, precision_recall_curve, average_precision_score
from sklearn.metrics import auc, roc_auc_score, roc_curve, f1_score, accuracy_score, recall_score, precision_score, brier_score_loss
import numpy as np

In [2]:
BASE_DIR = '../../../'

In [3]:
# get the fusion IDs
fusion_ids = pd.read_csv(BASE_DIR + '/data/fusion_ids_full_cohort.csv')
fusion_ids.shape, fusion_ids['label'].value_counts()

((806, 2),
 label
 0.0    630
 1.0    176
 Name: count, dtype: int64)

In [4]:
fusion_ids = fusion_ids.drop_duplicates()
fusion_ids = fusion_ids.reset_index(drop=True)
fusion_ids.shape

(681, 2)

In [5]:
fusion_ids['label'].value_counts()

label
0.0    559
1.0    122
Name: count, dtype: int64

In [6]:
df_test = pd.read_csv(BASE_DIR + '/data/test_data_full_cohort.csv')
df_test.shape, df_test['label'].value_counts()


((160, 3),
 label
 0.0    128
 1.0     32
 Name: count, dtype: int64)

In [7]:
df_test = df_test.drop_duplicates(subset=['id', 'label'])
df_test = df_test.reset_index(drop=True)
df_test.shape, df_test['label'].value_counts()

((135, 3),
 label
 0.0    111
 1.0     24
 Name: count, dtype: int64)

In [8]:
df_metadata = pd.read_csv('../../../data/all_file_user_metadata_new.csv')
df_metadata['Participant_ID'] = df_metadata['Participant_ID'].apply(
    lambda x: x[20:] if str(x).startswith('20') else x
)
df_metadata = df_metadata.drop_duplicates(subset=['Participant_ID'])

In [9]:
fusion_ids_carriers = fusion_ids[fusion_ids['label'] == 1]['id'].to_list()
len(fusion_ids_carriers)

122

In [10]:
df_metadata_carriers = df_metadata[df_metadata['Participant_ID'].isin(fusion_ids_carriers)]
df_metadata_carriers.reset_index(drop=True, inplace=True)
df_metadata_carriers.shape

(122, 15)

In [11]:
df_metadata_carriers.head()

,Filename,Protocol,Participant_ID,Task,Duration,FPS,Frame_Height,Frame_Width,gender,age,race,ethnicity,pd,dob,time_mdsupdrs
0,2021-11-03T15%3A45%3A34.633Z_b2n6WaQ74cU7Kz1H7...,ValorPD,b2n6WaQ74cU7Kz1H77OrBP9IYIZ2,disgust,6.369000,1000.0,720.0,1280.0,female,NaN,NaN,NaN,no,12/17/1982,NaN
1,2021-11-03T15%3A48%3A31.350Z_ILxMqv2XqlQGLGGBk...,ValorPD,ILxMqv2XqlQGLGGBksmpxpev1r23,disgust,4.040000,25.0,480.0,640.0,female,NaN,NaN,NaN,no,8/25/1970,NaN
2,2021-11-03T19%3A47%3A34.495Z_9QyLPApLvgPHnsosE...,ValorPD,9QyLPApLvgPHnsosEVwzEcr5pWs2,disgust,7.866667,30.0,480.0,640.0,female,NaN,NaN,NaN,no,4/22/1956,NaN
3,2021-11-10T15%3A16%3A24.385Z_pTVDr16JdWbkfgTVz...,ValorPD,pTVDr16JdWbkfgTVz6AHWLjUA3r2,disgust,7.733333,30.0,480.0,640.0,male,NaN,NaN,NaN,no,1/23/1946,NaN
4,2021-11-10T18%3A46%3A18.020Z_U3PnooCmSRTcGSDqU...,ValorPD,U3PnooCmSRTcGSDqUuHJZ3JKrvv1,disgust,8.495000,1000.0,480.0,640.0,male,NaN,NaN,NaN,no,11/20/1991,NaN


In [12]:
df_metadata_non_carriers = df_metadata[df_metadata['Participant_ID'].isin(fusion_ids['id'].to_list())]
df_metadata_non_carriers = df_metadata_non_carriers[~df_metadata_non_carriers['Participant_ID'].isin(fusion_ids_carriers)]

df_metadata_non_carriers.reset_index(drop=True, inplace=True)
df_metadata_non_carriers.shape

(559, 15)

In [13]:
carrier_demography = pd.read_csv('../../../data/valor_auth_table_demography.csv')
carrier_demography.head()

,record_id,sex,participating_year,dob,age,ethnicity,race,id
0,1,Female,2018,1963-10-17,55,Not Hispanic/Latino,White,HLOH7DSsvkVEd3JYgQVaiuc4oO12
1,2,Female,2019,1963-06-15,56,Hispanic/Latino,White,W4XvnRfOT5cvq7a8yxixshoBMVV2
2,3,Female,2019,1949-02-20,70,Not Hispanic/Latino,White,JLK7yz0KO3MGrw4abS5bVOJBn212
3,5,Male,2019,1961-03-26,58,Not Hispanic/Latino,White,Ep8qC4NnC8Xw0ieZxbklHIzrqzq1
4,6,Male,2018,1942-08-13,76,Not Hispanic/Latino,White,vr0epBbRqEeCKsNeYIXbhf9jFkR2


In [14]:
demography_dict = carrier_demography.set_index('id')[['age', 'race', 'sex']].to_dict('index')  

In [15]:
df_metadata_carriers['age'] = df_metadata_carriers['Participant_ID'].map(lambda x: demography_dict.get(x, {}).get('age'))
df_metadata_carriers['race'] = df_metadata_carriers['Participant_ID'].map(lambda x: demography_dict.get(x, {}).get('race'))
df_metadata_carriers['gender'] = df_metadata_carriers['Participant_ID'].map(lambda x: demography_dict.get(x, {}).get('sex'))

/tmp/ipykernel_216780/910065368.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_metadata_carriers['age'] = df_metadata_carriers['Participant_ID'].map(lambda x: demography_dict.get(x, {}).get('age'))
/tmp/ipykernel_216780/910065368.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_metadata_carriers['race'] = df_metadata_carriers['Participant_ID'].map(lambda x: demography_dict.get(x, {}).get('race'))
/tmp/ipykernel_216780/910065368.py:3: SettingWithCopyWarning: 
A value is trying to be set on a co

In [16]:
df_metadata_carriers['gender'].value_counts(dropna=False)

gender
Female    71
Male      51
Name: count, dtype: int64

In [17]:
import pandas as pd
import numpy as np

# Define the bin edges
# -np.inf captures everything below 20
# np.inf captures everything above 80
bins = [-np.inf, 20, 30, 40, 50, 60, 70, 80, np.inf]

# Define the labels for each slab
labels = ['<20', '20-30', '30-40', '40-50', '50-60', '60-70', '70-80', '>80']

# Create the new column 'age_group'
df_metadata_carriers['age_group'] = pd.cut(
    df_metadata_carriers['age'], 
    bins=bins, 
    labels=labels, 
    right=False # right=False means bins are [20, 30) i.e., 20 is included, 30 is not
)

# Check the distribution
print(df_metadata_carriers['age_group'].value_counts().sort_index())

age_group
<20       0
20-30    11
30-40    24
40-50    10
50-60    28
60-70    39
70-80    10
>80       0
Name: count, dtype: int64


/tmp/ipykernel_216780/2061888770.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_metadata_carriers['age_group'] = pd.cut(


In [18]:
df_metadata_carriers['race_normalized'] = df_metadata_carriers['race']
df_metadata_carriers['race_normalized'].value_counts(dropna=False)

/tmp/ipykernel_216780/2043432393.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_metadata_carriers['race_normalized'] = df_metadata_carriers['race']


race_normalized
White          116
Unavailable      6
Name: count, dtype: int64

In [19]:
df_metadata_non_carriers['gender'] = df_metadata_non_carriers['gender'].apply(lambda x: x.capitalize())
df_metadata_non_carriers['gender'].value_counts(dropna=False)

gender
Female    311
Male      248
Name: count, dtype: int64

In [20]:
import pandas as pd
import numpy as np

# Define the bin edges
# -np.inf captures everything below 20
# np.inf captures everything above 80
bins = [-np.inf, 20, 30, 40, 50, 60, 70, 80, np.inf]

# Define the labels for each slab
labels = ['<20', '20-30', '30-40', '40-50', '50-60', '60-70', '70-80', '>80']

# Create the new column 'age_group'
df_metadata_non_carriers['age_group'] = pd.cut(
    df_metadata_non_carriers['age'], 
    bins=bins, 
    labels=labels, 
    right=False # right=False means bins are [20, 30) i.e., 20 is included, 30 is not
)

# Check the distribution
print(df_metadata_non_carriers['age_group'].value_counts(dropna=False).sort_index())

age_group
<20        5
20-30     28
30-40     19
40-50     17
50-60    120
60-70    231
70-80     78
>80        4
NaN       57
Name: count, dtype: int64


In [21]:
# Create the 'race_normalized' column
df_metadata_non_carriers['race_normalized'] = df_metadata_non_carriers['race'].apply(
    lambda x: 'White' if 'white' in str(x).lower() else 'Non_white'
)

# Check the new distribution
print(df_metadata_non_carriers['race_normalized'].value_counts(dropna=False))

race_normalized
White        472
Non_white     87
Name: count, dtype: int64


In [22]:
import pandas as pd
import numpy as np
import math

# ==========================================
# CONFIGURATION
# ==========================================
TARGET_RATIO = 1.5  # Target: 183 controls total
RANDOM_SEED = 42

# Define the Match Levels
PASS_1_COLS = ['gender', 'race_normalized', 'age_group']
PASS_2_COLS = ['gender', 'race_normalized']
PASS_3_COLS = ['gender']

# ==========================================
# PREPARATION & FIXES
# ==========================================
# FIX 1: Explicitly copy the dataframes to avoid SettingWithCopyWarning
df_metadata_carriers = df_metadata_carriers.copy()
df_metadata_non_carriers = df_metadata_non_carriers.copy()

# Shuffle carriers to randomize who gets 2 matches
carriers = df_metadata_carriers.sample(frac=1, random_state=RANDOM_SEED).copy()
control_pool = df_metadata_non_carriers.copy()
control_pool['used'] = False 

matched_control_indices = []
match_quality_stats = {'Pass 1 (Exact)': 0, 'Pass 2 (Relaxed)': 0, 'Pass 3 (Gender Only)': 0}

# ACCUMULATOR: Must be defined OUTSIDE the loop
cumulative_target = 0.0

print(f"Starting Dynamic Matching (Target Ratio 1:{TARGET_RATIO})...")
print(f"Goal: {int(len(carriers) * TARGET_RATIO)} Controls for {len(carriers)} Carriers.")

# Debug Counters
count_needed_1 = 0
count_needed_2 = 0

# ==========================================
# MATCHING LOOP
# ==========================================
for idx, carrier in carriers.iterrows():
    
    # ---------------------------------------------------------
    # CALCULATE NEED (The Alternator Logic)
    # ---------------------------------------------------------
    prev_integer_target = int(cumulative_target)
    cumulative_target += TARGET_RATIO
    current_integer_target = int(cumulative_target)
    
    needed = current_integer_target - prev_integer_target
    
    # Debug tracking
    if needed == 1: count_needed_1 += 1
    if needed == 2: count_needed_2 += 1
    
    if needed == 0: continue

    matches_found_for_this_carrier = []
    
    # ---------------------------------------------------------
    # PASS 1: STRICT MATCH (Gender + Race + Age Group)
    # ---------------------------------------------------------
    if needed > 0:
        strict_mask = (
            (control_pool['gender'] == carrier['gender']) & 
            (control_pool['race_normalized'] == carrier['race_normalized']) & 
            (control_pool['age_group'] == carrier['age_group']) & 
            (control_pool['used'] == False)
        )
        candidates = control_pool[strict_mask]
        
        if len(candidates) > 0:
            n_to_pick = min(len(candidates), needed)
            picked = candidates.sample(n=n_to_pick, random_state=RANDOM_SEED)
            
            matches_found_for_this_carrier.extend(picked.index.tolist())
            control_pool.loc[picked.index, 'used'] = True
            needed -= n_to_pick
            match_quality_stats['Pass 1 (Exact)'] += n_to_pick

    # ---------------------------------------------------------
    # PASS 2: RELAXED MATCH (Gender + Race) - Ignore Age
    # ---------------------------------------------------------
    if needed > 0:
        relaxed_mask = (
            (control_pool['gender'] == carrier['gender']) & 
            (control_pool['race_normalized'] == carrier['race_normalized']) & 
            (control_pool['used'] == False)
        )
        candidates = control_pool[relaxed_mask]
        
        if len(candidates) > 0:
            n_to_pick = min(len(candidates), needed)
            picked = candidates.sample(n=n_to_pick, random_state=RANDOM_SEED)
            
            matches_found_for_this_carrier.extend(picked.index.tolist())
            control_pool.loc[picked.index, 'used'] = True
            needed -= n_to_pick
            match_quality_stats['Pass 2 (Relaxed)'] += n_to_pick

    # ---------------------------------------------------------
    # PASS 3: NUCLEAR MATCH (Gender Only) - Ignore Age & Race
    # ---------------------------------------------------------
    if needed > 0:
        nuclear_mask = (
            (control_pool['gender'] == carrier['gender']) & 
            (control_pool['used'] == False)
        )
        candidates = control_pool[nuclear_mask]
        
        if len(candidates) > 0:
            n_to_pick = min(len(candidates), needed)
            picked = candidates.sample(n=n_to_pick, random_state=RANDOM_SEED)
            
            matches_found_for_this_carrier.extend(picked.index.tolist())
            control_pool.loc[picked.index, 'used'] = True
            needed -= n_to_pick
            match_quality_stats['Pass 3 (Gender Only)'] += n_to_pick

    matched_control_indices.extend(matches_found_for_this_carrier)

# ==========================================
# FINALIZE DATASET
# ==========================================
df_matched_controls = df_metadata_non_carriers.loc[matched_control_indices].copy()

# Add status labels safely using .loc
df_metadata_carriers.loc[:, 'status'] = 'Carrier'
df_matched_controls.loc[:, 'status'] = 'Control'

df_final_matched = pd.concat([df_metadata_carriers, df_matched_controls], axis=0).reset_index(drop=True)

# ==========================================
# REPORTING
# ==========================================
print("-" * 40)
print("FINAL MATCHING SUMMARY")
print("-" * 40)
print(f"Target Ratio:         1 : {TARGET_RATIO}")
print(f"Total Carriers:       {len(carriers)}")
print(f"Requests for 1 match: {count_needed_1}")
print(f"Requests for 2 match: {count_needed_2}")
print(f"Expected Controls:    {(count_needed_1 * 1) + (count_needed_2 * 2)}")
print(f"Actual Controls:      {len(df_matched_controls)}")
print(f"Actual Final Ratio:   1 : {len(df_matched_controls) / len(carriers):.2f}")
print("-" * 40)
print("QUALITY BREAKDOWN:")
for key, value in match_quality_stats.items():
    pct = (value / len(df_matched_controls)) * 100 if len(df_matched_controls) > 0 else 0
    print(f"  - {key:<20}: {value} ({pct:.1f}%)")

Starting Dynamic Matching (Target Ratio 1:1.5)...
Goal: 183 Controls for 122 Carriers.
----------------------------------------
FINAL MATCHING SUMMARY
----------------------------------------
Target Ratio:         1 : 1.5
Total Carriers:       122
Requests for 1 match: 61
Requests for 2 match: 61
Expected Controls:    183
Actual Controls:      183
Actual Final Ratio:   1 : 1.50
----------------------------------------
QUALITY BREAKDOWN:
  - Pass 1 (Exact)      : 142 (77.6%)
  - Pass 2 (Relaxed)    : 33 (18.0%)
  - Pass 3 (Gender Only): 8 (4.4%)


In [23]:
# Function to print clean distribution tables
def print_distribution(df, col_name):
    # Calculate counts and percentages
    dist = df.groupby('status')[col_name].value_counts(normalize=True).unstack().T
    dist = dist * 100 # Convert to percentage
    
    # Format for display
    print(f"\n=== {col_name.upper()} DISTRIBUTION (%) ===")
    print(dist.round(2)) # Round to 2 decimal places
    
    # Calculate difference (bias check)
    if 'Carrier' in dist.columns and 'Control' in dist.columns:
        diff = (dist['Carrier'] - dist['Control']).abs().mean()
        print(f"--> Average Discrepancy: {diff:.2f}%")

# 1. Gender Distribution
print_distribution(df_final_matched, 'gender')

# 2. Race Distribution
print_distribution(df_final_matched, 'race_normalized')

# 3. Age Group Distribution
print_distribution(df_final_matched, 'age_group')

# 4. Final Count Check
print("\n=== FINAL COUNTS ===")
print(df_final_matched['status'].value_counts())


=== GENDER DISTRIBUTION (%) ===
status  Carrier  Control
gender                  
Female     58.2    57.38
Male       41.8    42.62
--> Average Discrepancy: 0.82%

=== RACE_NORMALIZED DISTRIBUTION (%) ===
status           Carrier  Control
race_normalized                  
Non_white            NaN     1.64
Unavailable         4.92      NaN
White              95.08    98.36
--> Average Discrepancy: 3.28%

=== AGE_GROUP DISTRIBUTION (%) ===
status     Carrier  Control
age_group                  
<20           0.00     0.00
20-30         9.02     6.82
30-40        19.67     6.25
40-50         8.20     6.25
50-60        22.95    26.70
60-70        31.97    41.48
70-80         8.20    11.93
>80           0.00     0.57
--> Average Discrepancy: 4.39%

=== FINAL COUNTS ===
status
Control    183
Carrier    122
Name: count, dtype: int64


In [24]:
df_final_matched.head()

,Filename,Protocol,Participant_ID,Task,Duration,FPS,Frame_Height,Frame_Width,gender,age,race,ethnicity,pd,dob,time_mdsupdrs,age_group,race_normalized,status
0,2021-11-03T15%3A45%3A34.633Z_b2n6WaQ74cU7Kz1H7...,ValorPD,b2n6WaQ74cU7Kz1H77OrBP9IYIZ2,disgust,6.369000,1000.0,720.0,1280.0,Female,38.0,White,NaN,no,12/17/1982,NaN,30-40,White,Carrier
1,2021-11-03T15%3A48%3A31.350Z_ILxMqv2XqlQGLGGBk...,ValorPD,ILxMqv2XqlQGLGGBksmpxpev1r23,disgust,4.040000,25.0,480.0,640.0,Female,49.0,White,NaN,no,8/25/1970,NaN,40-50,White,Carrier
2,2021-11-03T19%3A47%3A34.495Z_9QyLPApLvgPHnsosE...,ValorPD,9QyLPApLvgPHnsosEVwzEcr5pWs2,disgust,7.866667,30.0,480.0,640.0,Female,63.0,White,NaN,no,4/22/1956,NaN,60-70,White,Carrier
3,2021-11-10T15%3A16%3A24.385Z_pTVDr16JdWbkfgTVz...,ValorPD,pTVDr16JdWbkfgTVz6AHWLjUA3r2,disgust,7.733333,30.0,480.0,640.0,Male,73.0,White,NaN,no,1/23/1946,NaN,70-80,White,Carrier
4,2021-11-10T18%3A46%3A18.020Z_U3PnooCmSRTcGSDqU...,ValorPD,U3PnooCmSRTcGSDqUuHJZ3JKrvv1,disgust,8.495000,1000.0,480.0,640.0,Male,27.0,White,NaN,no,11/20/1991,NaN,20-30,White,Carrier


In [ ]:
# from sklearn.model_selection import train_test_split
# import os

# # CONFIGURATION
# ID_COLUMN = 'Participant_ID'  # Make sure this matches your column name (e.g., 'id' or 'Participant_ID')
# OUTPUT_DIR = '../../../data/' # Save in current directory

# # 1. First Split: Separate Train (60%) from the Rest (40%)
# # We stratify by 'status' to ensure equal ratios of Carriers/Controls in all sets
# train_df, temp_df = train_test_split(
#     df_final_matched, 
#     test_size=0.40, 
#     stratify=df_final_matched['status'], 
#     random_state=42
# )

# # 2. Second Split: Split the Rest (40%) into Dev (20%) and Test (20%)
# # Splitting the 40% in half gives us 20% total for each
# dev_df, test_df = train_test_split(
#     temp_df, 
#     test_size=0.50, 
#     stratify=temp_df['status'], 
#     random_state=42
# )

# # 3. Print Stats to Verify
# print("--- SPLIT STATISTICS ---")
# print(f"Train Set: {len(train_df)} ({len(train_df)/len(df_final_matched):.0%})")
# print(f"Dev Set:   {len(dev_df)} ({len(dev_df)/len(df_final_matched):.0%})")
# print(f"Test Set:  {len(test_df)} ({len(test_df)/len(df_final_matched):.0%})")
# print("-" * 30)
# print("Carrier Counts per Set:")
# print(f"Train: {train_df['status'].value_counts()['Carrier']}")
# print(f"Dev:   {dev_df['status'].value_counts()['Carrier']}")
# print(f"Test:  {test_df['status'].value_counts()['Carrier']}")

# # 4. Helper Function to Save IDs
# def save_ids(df, filename):
#     ids = df[ID_COLUMN].tolist()
#     path = os.path.join(OUTPUT_DIR, filename)
#     with open(path, 'w') as f:
#         for pid in ids:
#             f.write(f"{pid}\n")
#     print(f"Saved {len(ids)} IDs to {filename}")

# # 5. Save the files
# save_ids(train_df, 'train_set_participants_demo_matched.txt')
# save_ids(dev_df, 'dev_set_participants_demo_matched.txt')
# save_ids(test_df, 'test_set_participants_demo_matched.txt')

--- SPLIT STATISTICS ---
Train Set: 183 (60%)
Dev Set:   61 (20%)
Test Set:  61 (20%)
------------------------------
Carrier Counts per Set:
Train: 73
Dev:   24
Test:  25
Saved 183 IDs to train_set_participants.txt
Saved 61 IDs to dev_set_participants.txt
Saved 61 IDs to test_set_participants.txt
